<h1>Entrenamiento del Modelo MassBalanceMachine XGBoost - Ejemplo de la CB</h1>
<p style='text-align: justify;'>
En este notebook, simularemos el balance de masa superficial glaciar para la región de Perú utilizando un modelo personalizado de <a href='https://xgboost.readthedocs.io/en/stable/'>XGBoost</a>. El modelo XGBoost está diseñado con una función objetivo personalizada que genera predicciones mensuales basadas en datos observacionales agregados. Crearemos una instancia de <code>CustomXGBoostRegressor</code> y la entrenaremos usando esta función de pérdida personalizada con los datos de estacas de la CB, los cuales hemos preparado en notebooks anteriores. 

<p style='text-align: justify;'>
El flujo de trabajo incluye varios pasos clave:
</p>

<ol style="margin-left: 20px; padding-left: 0;">
    <li style="margin-bottom: 10px;">
        <p style='text-align: justify;'><strong>Carga y Preparación de Datos:</strong> Se crea un objeto <code>Dataloader</code> para gestionar la carga de datos y la creación de una división entre entrenamiento y prueba. Este objeto también maneja la generación de particiones de datos para la validación cruzada.</p>
    </li>
    <li style="margin-bottom: 10px;">
        <p style='text-align: justify;'><strong>Validación Cruzada y Entrenamiento del Modelo:</strong> Utilizando las técnicas de validación cruzada de Scikit-learn, exploramos diferentes hiperparámetros y entrenamos el modelo con las particiones de datos preparadas. Este enfoque asegura una evaluación robusta y ayuda en la selección de parámetros adecuados.</p>
    </li>
    <li style="margin-bottom: 10px;">
        <p style='text-align: justify;'><strong>Predicciones Agregadas:</strong> Después del entrenamiento, mostraremos las predicciones mensuales agregadas generadas por el modelo para visualizar y analizar los resultados.</p>
    </li>
    <li style="margin-bottom: 10px;">
        <p style='text-align: justify;'><strong>Evaluación del Modelo:</strong> Finalmente, se evalúa el desempeño del modelo en el conjunto de prueba, proporcionando información sobre su precisión predictiva para el balance de masa glaciar.</p>
    </li>
</ol>


In [ ]:
import pandas as pd
import massbalancemachine as mbm
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
import random

random.seed(42)
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

In [ ]:
# Set a random seed:
data = pd.read_csv('./data/SouthernAndes_monthly_dataset.csv')
print('Number of winter and annual samples:', len(data))
display(data)

cfg = mbm.Config()

<h2>1. Crear el Conjunto de Datos de Entrenamiento y Prueba y las Divisiones de Datos para la Validación Cruzada</h2>
<p style='text-align: justify;'>
Primero, creamos un objeto <code>DataLoader</code>, que genera tanto los conjuntos de datos de entrenamiento como de prueba, así como las divisiones de datos necesarias para la validación cruzada. Para conservar memoria, el método <code>set_train_test_split</code> devuelve iteradores que contienen los índices para los conjuntos de entrenamiento y prueba. Estos índices se utilizan luego para recuperar los datos correspondientes para el entrenamiento y la prueba. A continuación, el método <code>get_cv_split</code> proporciona una lista que indica el número de particiones necesarias para la validación cruzada.
</p>

In [ ]:
# Crear un nuevo objeto DataLoader con las mediciones mensuales de datos de estacas.
dataloader = mbm.dataloader.DataLoader(cfg, data=data)
# Crear iteradores para entrenamiento y prueba. Los parámetros son opcionales. El valor por defecto de test_size es 0.3.
train_itr, test_itr = dataloader.set_train_test_split(test_size=0.3)

# Obtener todos los índices del conjunto de entrenamiento y prueba de una vez desde los iteradores. Una vez llamados, los iteradores quedan vacíos.
train_indices, test_indices = list(train_itr), list(test_itr)

# Obtener las características y los valores objetivo del conjunto de entrenamiento para los índices definidos anteriormente, que se usarán durante la validación cruzada.
df_X_train = data.iloc[train_indices]
y_train = df_X_train['POINT_BALANCE'].values

# Obtener el conjunto de prueba
df_X_test = data.iloc[test_indices]
y_test = df_X_test['POINT_BALANCE'].values

# Crear las divisiones para la validación cruzada basadas en el conjunto de entrenamiento. El valor por defecto del número de divisiones es 5.
splits = dataloader.get_cv_split(n_splits=5)

# Imprimir el tamaño de los conjuntos de entrenamiento y prueba
print(f"Size of training set: {len(train_indices)}")
print(f"Size of test set: {len(test_indices)}")

<h2>2. Crear un Modelo CustomXGBoostRegressor</h2>
<p style='text-align: justify;'>
A continuación, definimos los rangos de parámetros para cada parámetro de XGBoost. En el paso siguiente, utilizamos la validación cruzada para explorar estos rangos de parámetros y seleccionar la combinación que produzca la menor pérdida. Además, creamos un objeto <code>CustomXGBoostRegressor</code>.
</p>

In [ ]:
# Para cada uno de los parámetros de XGBoost, defina el rango de la cuadrícula
parameters = {
    'max_depth': [
        3,
        4,
        5,
        6,
    ],
    'learning_rate': [0.01, 0.1, 0.2, 0.3],
    'n_estimators': [100, 200, 300],
    'gamma': [0, 1]
}

In [ ]:
# Crear una instancia de CustomXGBoostRegressor
params_init = {"device": "cpu"}
custom_xgboost = mbm.models.CustomXGBoostRegressor(cfg, **params_init)

<h2>3. Entrenar el Modelo CustomXGBoostRegressor</h2>

<p style='text-align: justify; margin-bottom: 5px;'>
En la siguiente celda, comenzamos el entrenamiento de nuestro modelo utilizando <strong>GridSearchCV</strong> o <strong>RandomizedSearchCV</strong>:
</p>

<ul style="margin-left: 20px; padding-left: 0; margin-bottom: 5px;">
  <li style="margin-bottom: 10px;">
    <p style='text-align: justify;'><strong>GridSearchCV</strong> realiza una búsqueda exhaustiva entre todas las combinaciones posibles de parámetros para encontrar el mejor conjunto que ofrezca el rendimiento óptimo utilizando validación cruzada. Aunque este método es minucioso, suele ser lento y computacionalmente costoso.</p>
  </li>
  <li style="margin-bottom: 0px;">
    <p style='text-align: justify;'><strong>RandomizedSearchCV</strong>, por otro lado, selecciona un número fijo de combinaciones de parámetros de la distribución, lo que lo hace más eficiente en términos de tiempo y recursos computacionales, especialmente en espacios de hiperparámetros grandes. Sin embargo, este enfoque puede omitir algunas de las mejores combinaciones de parámetros que no sean seleccionadas aleatoriamente.</p>
  </li>
</ul>

<p style='text-align: justify;'>
Puedes elegir cualquiera de los dos métodos de entrenamiento. Ambos métodos utilizarán todos los núcleos de la CPU por defecto. Si deseas ajustar el número de núcleos utilizados, puedes cambiar el parámetro <code>num_jobs</code>.
</p>


In [ ]:
# Búsqueda en cuadrícula
# custom_xgboost.gridsearch(parameters=parameters, splits=splits, features=df_X_train, targets=y_train, num_jobs=-1)

# Búsqueda aleatoria, con n_iter como el número de parámetros muestreados. Compromiso entre la calidad de la solución y el tiempo de ejecución.
custom_xgboost.randomsearch(
    parameters=parameters,
    n_iter=20,
    splits=splits,
    features=df_X_train,
    targets=y_train,
)
best_params = params = custom_xgboost.param_search.best_params_
best_estimator = custom_xgboost.param_search.best_estimator_
print("Best parameters:\n", best_params)
print("Best score:\n", custom_xgboost.param_search.best_score_)

In [ ]:
plt.plot(custom_xgboost.param_search.cv_results_['mean_train_score'])
plt.plot(custom_xgboost.param_search.cv_results_['mean_test_score'])

<h4>3.1 Guardar la modelo entrenada</h4>

In [ ]:
custom_xgboost.save_model('model_xgb.pkl')

In [ ]:
#mbm.models.CustomXGBoostRegressor.load_model('model_xgb.pkl')

<h3>3.1 Mostrar las predicciones</h3>

In [ ]:
dict_glaciers = { "RGI60-17.03160": "Shiaparelli",
                "RGI60-17.04856": "Tyndall",
                "RGI60-17.00312": "Perito Moreno",
                "RGI60-17.04871": "Grey + Dickson",
                "RGI60-17.15831": "Exploradores"}
dict_glaciers_colors = { "Shiaparelli": "blue",
                "Tyndall": "purple",
                "Perito Moreno": "yellow",
                "Grey + Dickson": "green",
                "Exploradores": "red"}

def predVSTruth(grouped_ids, mae, rmse, title):
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    legend_xgb = "\n".join(
        (r"$\mathrm{MAE_{xgb}}=%.3f, \mathrm{RMSE_{xgb}}=%.3f$ " % (
            mae,
            rmse,
        ), ))

    marker_xgb = 'o'
    sns.scatterplot(grouped_ids,
                    x="target",
                    y="pred",
                    ax=ax,
                    alpha=0.3,
                    s= 100,
                    marker=marker_xgb,
                    hue="label",            # etiquetas del diccionario
                    palette=dict_glaciers_colors)

    ax.set_ylabel('Predicted PMB [m w.e.]', fontsize=20)
    ax.set_xlabel('Observed PMB [m w.e.]', fontsize=20)

    ax.text(0.03,
            0.98,
            legend_xgb,
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=20)
    ax.legend()
    # diagonal line
    pt = (0, 0)
    ax.axline(pt, slope=1, color="grey", linestyle="-", linewidth=0.2)
    ax.axvline(0, color="grey", linestyle="-", linewidth=0.2)
    ax.axhline(0, color="grey", linestyle="-", linewidth=0.2)
    ax.grid()
    ax.set_title(title, fontsize=20)
    plt.savefig("Prediccion.png", dpi=300, bbox_inches="tight")
    plt.tight_layout()

In [ ]:
# Configurar para usar la CPU en las predicciones:
xgb = best_estimator.set_params(device='cpu')

# Realizar predicciones en el conjunto de prueba
features_test, metadata_test = xgb._create_features_metadata(df_X_test)
y_pred = xgb.predict(features_test)

# Realizar predicciones agregadas por ID de medición:
y_pred_agg = xgb.aggrPredict(metadata_test, features_test)

# Calcular métricas
score = xgb.score(df_X_test, y_test)  # negativo
mse, rmse, mae, pearson_corr, r2, bias = xgb.evalMetrics(metadata_test, y_pred, y_test)

# Agregar predicciones a escala anual o invernal:
df_pred = df_X_test.copy()
df_pred['target'] = y_test

grouped_ids = df_pred.groupby('ID').agg({'target': 'mean', 'RGIId':'first'})
grouped_ids["label"] = grouped_ids["RGIId"].map(dict_glaciers)
grouped_ids['pred'] = y_pred_agg

predVSTruth(grouped_ids, mae, rmse, title='XGBoost on Patagonia')

In [ ]:
# Configurar para usar la CPU en las predicciones:
xgb = best_estimator.set_params(device='cpu')

# Realizar predicciones en el conjunto de prueba
features_test, metadata_test = xgb._create_features_metadata(data)
y_pred = xgb.predict(features_test)
y_test = data['POINT_BALANCE'].values

# Realizar predicciones agregadas por ID de medición:
y_pred_agg = xgb.aggrPredict(metadata_test, features_test)

# Calcular métricas
score = xgb.score(data, y_test)  # negativo
mse, rmse, mae, pearson_corr, r2, bias = xgb.evalMetrics(metadata_test, y_pred, y_test)

# Agregar predicciones a escala anual o invernal:
df_pred = data.copy()
df_pred['target'] = y_test

grouped_ids = df_pred.groupby('ID').agg({'target': 'mean', 'RGIId':'first'})
grouped_ids["label"] = grouped_ids["RGIId"].map(dict_glaciers)
grouped_ids['pred'] = y_pred_agg

predVSTruth(grouped_ids, mae, rmse, title='XGBoost on Patagonia')